<a href="https://colab.research.google.com/github/guedesrj/analise-sentimentos-americanas/blob/main/analise_sentimentos_americanas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
# Importacoes

import pandas as pd, numpy as np, re, json
import requests

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
try:
  import nltk
except ImportError:
    print("NLTK not found. Installing NLTK...")
    %pip install nltk
    import nltk
from nltk.corpus import stopwords

In [63]:
#Importar base de dados

url = "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/main/B2W-Reviews01.csv"
filename = "b2w.csv"

print(f"Downloading dataset from {url}...")
response = requests.get(url)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

with open(filename, "wb") as f:
    f.write(response.content)

print(f"Dataset saved as {filename}")

Dataset saved as b2w.csv


In [65]:
# Importar StopWords

nltk.download("stopwords")

# stopwords em portugues
stop_words = set(stopwords.words("portuguese"))

print("NLTK stopwords for Portuguese loaded successfully.")

NLTK stopwords for Portuguese loaded successfully.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [66]:
# Carregar os dados
print("Carregando dados...")

df = pd.read_csv("b2w.csv",usecols=["review_text", "overall_rating"])

print(f"Total de registros: {len(df)}")

# Limpeza dos dados duplicados e ausentes
df = df.dropna(subset=["review_text"])
df["overall_rating"] = pd.to_numeric(df["overall_rating"], errors="coerce")
df = df.dropna(subset=["overall_rating"])
df = df.drop_duplicates(subset=["review_text"])

print(f"Apos remocao de nulos/duplicados: {len(df)}")

dist = df["overall_rating"].value_counts().sort_index()
print("Distribuicao de notas:\n", dist)

# Rotulo binario: 1-2 negativo, 4-5 positivo e 3 descartado

df = df[df["overall_rating"] != 3].copy()
df["label"] = (df["overall_rating"] >= 4).astype(int)
print(f"Apos descartar nota 3: {len(df)}")
print("Distribuicao de classes:\n", df["label"].value_counts())

# Preprocessamento dos dados

def preprocessar(texto):
  texto = texto.lower()
  texto = re.sub(r"[^a-zà-ú\s]", " ", texto)
  palavras = texto.split()
  palavras = [palavra for palavra in palavras if palavra not in stop_words]
  texto = " ".join(palavras)
  return texto
print("Pre-processando textos...")

df["review_text_processed"] = df["review_text"].apply(preprocessar)

df = df[df["review_text_processed"].str.len() > 0]
print(f"Apos remocao de textos vazios: {len(df)}")
df.head()

Carregando dados...
Total de registros: 132373
Apos remocao de nulos/duplicados: 126724
Distribuicao de notas:
 overall_rating
1    24608
2     7992
3    15849
4    31786
5    46489
Name: count, dtype: int64
Apos descartar nota 3: 110875
Distribuicao de classes:
 label
1    78275
0    32600
Name: count, dtype: int64
Pre-processando textos...
Apos remocao de textos vazios: 110798


,overall_rating,review_text,label,review_text_processed
0,4,Estou contente com a compra entrega rápida o ú...,1,contente compra entrega rápida único problema ...
1,4,"Por apenas R$1994.20,eu consegui comprar esse ...",1,apenas r consegui comprar lindo copo acrílico
2,4,SUPERA EM AGILIDADE E PRATICIDADE OUTRAS PANEL...,1,supera agilidade praticidade outras panelas el...
3,4,MEU FILHO AMOU! PARECE DE VERDADE COM TANTOS D...,1,filho amou parece verdade tantos detalhes têm
4,5,"A entrega foi no prazo, as americanas estão de...",1,entrega prazo americanas parabéns smart tv boa...


In [67]:
# Divisao de Treino e Teste

X_treino, X_teste, y_treino, y_teste = train_test_split(
    df['review_text_processed'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
    )
print(f"Treino: {len(X_treino)} | Teste: {len(X_teste)}")

Treino: 88638 | Teste: 22160


In [68]:
# Vetorizacao TF-IDF
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=None,
    ngram_range=(1, 2),
    min_df=5,
    max_features=50000
)

X_treino_tfidf = vectorizer.fit_transform(X_treino)

X_teste_tfidf = vectorizer.transform(X_teste)

print(f"Quantidade de palavras no vocabulário:{len(vectorizer.vocabulary_)}")
print(f"Formato da matriz TF-IDF: {X_treino_tfidf.shape}")

Quantidade de palavras no vocabulário:34530
Formato da matriz TF-IDF: (88638, 34530)


In [69]:
# Modelo e treinamento
results = {}

for name, model in [("Regressao Logistica", LogisticRegression(max_iter=1000, class_weight="balanced")),("Naive Bayes", MultinomialNB())]:
    model.fit(X_treino_tfidf, y_treino)
    pred = model.predict(X_teste_tfidf) # Changed X_teste to X_teste_tfidf
    acc = accuracy_score(y_teste, pred)
    p, r, f1, _ = precision_recall_fscore_support(y_teste, pred, average="macro")
    cm = confusion_matrix(y_teste, pred)
    results[name] = dict(acc=round(acc,4), prec=round(p,4), rec=round(r,4), f1=round(f1,4), cm=cm.tolist())
    print(f"\n=== {name} ===")
    print(classification_report(y_teste, pred, target_names=["Negativo","Positivo"], digits=4))
    print("Matriz de confusao:\n", cm)


=== Regressao Logistica ===
              precision    recall  f1-score   support

    Negativo     0.8517    0.9452    0.8960      6519
    Positivo     0.9761    0.9314    0.9532     15641

    accuracy                         0.9355     22160
   macro avg     0.9139    0.9383    0.9246     22160
weighted avg     0.9395    0.9355    0.9364     22160

Matriz de confusao:
 [[ 6162   357]
 [ 1073 14568]]

=== Naive Bayes ===
              precision    recall  f1-score   support

    Negativo     0.8913    0.8751    0.8831      6519
    Positivo     0.9483    0.9555    0.9519     15641

    accuracy                         0.9319     22160
   macro avg     0.9198    0.9153    0.9175     22160
weighted avg     0.9316    0.9319    0.9317     22160

Matriz de confusao:
 [[ 5705   814]
 [  696 14945]]


In [71]:
# Gerar dicionario com os dados dos resultados e gerar json

meta = dict(total_bruto=132373, apos_limpeza=int(len(df)), treino=int(len(X_treino)), teste=int(len(X_teste)),
            vocab=int(X_treino_tfidf.shape[1]), dist_notas={int(k): int(v) for k, v in dist.items()},
            classes={int(k): int(v) for k, v in df["label"].value_counts().items()})
json.dump(dict(meta=meta, results=results), open("resultados.json","w"), indent=2)
print("\nSalvo em resultados.json")


Salvo em resultados.json


In [74]:
#Exportar p formato json

import json, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
d = json.load(open('resultados.json'))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
notas = d['meta']['dist_notas']
axes[0].bar([str(k) for k in sorted(notas, key=int)], [notas[k] for k in sorted(notas, key=int)], color=['#c0392b','#e67e22','#95a5a6','#27ae60','#145a32'])
axes[0].set_title('(a) Distribuição das notas no corpus')
axes[0].set_xlabel('Nota (estrelas)'); axes[0].set_ylabel('Quantidade de avaliações')
cm = np.array(d['results']['Regressao Logistica']['cm'])
im = axes[1].imshow(cm, cmap='Blues')
axes[1].set_title('(b) Matriz de confusão – Regressão Logística')
axes[1].set_xticks([0,1]); axes[1].set_yticks([0,1])
axes[1].set_xticklabels(['Negativo','Positivo']); axes[1].set_yticklabels(['Negativo','Positivo'])
axes[1].set_xlabel('Classe predita'); axes[1].set_ylabel('Classe real')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f'{cm[i,j]:,}'.replace(',','.'), ha='center', va='center', color='white' if cm[i,j]>7000 else 'black', fontsize=12)
plt.tight_layout()
plt.savefig('figura1.png', dpi=150, bbox_inches='tight')
plt.show()
print('ok')

ok


In [73]:
# Gerar aquivos com os resultados

import matplotlib.pyplot as plt
import seaborn as sns

# Gerar matrix de confusao
for name, data in results.items():
    cm = np.array(data['cm'])
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negativo', 'Positivo'], yticklabels=['Negativo', 'Positivo'])
    plt.title(f'Matriz de Confusão para {name}')
    plt.xlabel('Predito')
    plt.ylabel('Real')
    plt.savefig(f'confusion_matrix_{name.replace(' ', '_')}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Gerar distribuicao dos resultados antes do processamento
plt.figure(figsize=(8, 6))
sns.barplot(x=dist.index, y=dist.values, palette='magma')
plt.title('Distribuição das Notas Originais')
plt.xlabel('Nota')
plt.ylabel('Contagem')
plt.savefig('rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

/tmp/ipykernel_335/3111055441.py:19: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=dist.index, y=dist.values, palette='magma')
